In [ ]:
# For a complete daily bias correction model, we use two alternatives
# chain the best classification model (script 4B_3) with the best regression model (script 4B_4).
# create a new single model that regresses bias correction in a single step
# compare test performance of the two alternatives
# retrain the best model with feature elimination, save


#FIX: currently the RFECV eliminates feature using a simpler parameters
# and both RFECV and RSCV use with less cross fold iteration to avoid performance bottlneck


In [2]:
# Import libraries
import os
import numpy as np
import pandas as pd
import joblib
import ast 
import gc
import json


from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold, KFold

# Paths
train_df_path = r"..\Data\Processed\Bias_correction_daily\Prepared_dataframe\final_train_df.parquet"
test_df_path = r"..\Data\Processed\Bias_correction_daily\Prepared_dataframe\final_test_df.parquet"
regression_model_path = r"..\Data\Processed\Bias_correction_daily\Regressor_model\saved_regressors"
classifier_model_path = r"..\Data\Processed\Bias_correction_daily\Classifier_model\saved_classifiers"
composite_model_path = r"..\Data\Processed\Bias_correction_daily\Composite_model"
final_model_path = r"..\Data\Processed\Bias_correction_daily\Final_model"
os.makedirs(final_model_path, exist_ok=True)

os.makedirs(composite_model_path, exist_ok=True)

target_var_regression = 'insitu_prec'
target_var_classification = 'insitu_prec_class'  # Assuming the same variable for both tasks
random_state_num = 64
n_cores =  4        
predictor_features = ['y', 'x', 
        'era5l', 'chirps', 'har', 
        'persiann_prec', 'mswep_prec', 
        'gldas_air_temp', 'gldas_lwdown', 'gldas_specific_humidity',
        'gldas_surface_pressure', 'gldas_swdown', 'gldas_wind_speed',
#        'insitu_kriged', 
        'cluster', 'season', 'season_num',
        'elev_mean', 'elev_std', 'elev_max', 'elev_min', 'slope_mean', 
        'aspect_mean', 'elev_smoothed', 'fetch_1000m_dist', 'fetch_1000m_dir',
        'fetch_500m_dist', 'fetch_500m_dir', 'local_relief', 'doy_sin', 'doy_cos']

In [3]:
SEASONS = {
    'winter':       [12, 1, 2],
    'pre_monsoon':  [3, 4, 5],
    'monsoon':      [6, 7, 8, 9],
    'post_monsoon': [10, 11],
}

def get_season(month):
    for s, months in SEASONS.items():
        if month in months:
            return s
    return None

SEASONS_NUM = {
    1: 'winter',
    2: 'pre_monsoon',
    3: 'monsoon',
    4: 'post_monsoon'
}

def get_season_num(season_str):
    for s_n, s_name in SEASONS_NUM.items():
        if s_name == season_str:
            return s_n
    return None

# Helper functions
def get_param_grids():
    param_dist_rf = {
        'n_estimators':      [500, 1000, 2000, 3000, 5000],
        'max_features':      ['sqrt', 'log2'],
        'max_depth':         [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
    }
    param_dist_xgb = {
        'n_estimators':     [100, 500, 1000, 2000, 3000],
        'max_depth':        [3, 5, 7, 10],
        'learning_rate':    [0.01, 0.05, 0.1, 0.2],
        'subsample':        [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma':            [0, 0.1, 0.5],
    }
    
    return param_dist_rf, param_dist_xgb

def get_param_grids_increased():
    param_dist_rf = {
        'n_estimators':      [7000, 10000, 15000],
        'max_features':      ['sqrt', 'log2'],
        'max_depth':         [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
    }
    param_dist_xgb = {
        'n_estimators':     [5000, 7000, 10000],
        'max_depth':        [3, 5, 7, 10],
        'learning_rate':    [0.01, 0.05, 0.1, 0.2],
        'subsample':        [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma':            [0, 0.1, 0.5],
    }
    
    return param_dist_rf, param_dist_xgb

def compute_metrics_classification(y_true, y_pred, label):
    """ Compute categorical metrices"""
    #define event -  wet day is when precipitation is greater than 0.1
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred) 
    print(f"[{label}] | accuracy = {accuracy:.02f}| f1-score = {f1:.02f}| n = {len(y_true)}")
    return {'accuracy': round(accuracy, 3), 'f1-score': round(f1, 3), 'n': len(y_true)}

def compute_metrics_regression(y_true, y_pred, label):
    """Compute RMSE, MAE, R² and print with label."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"    [{label}]  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  n={len(y_true)}")
    return {'rmse': round(rmse, 4), 'mae': round(mae, 4), 'r2': round(r2, 4),
            'n': len(y_true)}

In [4]:
# 1. Load and format dfs
train_df = pd.read_parquet(train_df_path)
test_df = pd.read_parquet(test_df_path)

seasons_train_da = train_df['month'].apply(get_season)
train_df['season'] = seasons_train_da
seasons_test_da = test_df['month'].apply(get_season)
test_df['season'] = seasons_test_da

train_df['season_num'] = train_df['season'].apply(get_season_num)
test_df['season_num'] = test_df['season'].apply(get_season_num)

cluster_list = sorted(test_df['cluster'].unique().tolist())
season_list = list(SEASONS)

In [5]:
# # Train a single regression model for each season and predict on test data
# for season in season_list:
#     season_train_df = train_df[train_df['season'] == season].copy()
#     X_train = season_train_df[predictor_features].drop(columns=['season', 'season_num'], errors='ignore').copy()
#     y_train = season_train_df[target_var_regression].copy()

#     if season in ['monsoon']:
#         param_dist_rf, _ = get_param_grids_increased()
#         _, param_dist_xgb = get_param_grids()
#         for model_type in ['rf', 'xgb']:
#             if model_type == 'rf':
#                 base_model = Pipeline([
#                     ('imputer', SimpleImputer(strategy = 'median')),
#                     ('regressor', RandomForestRegressor(random_state = random_state_num,
#                         n_jobs = n_cores))
#                 ])
#                 param_dist = {f'regressor__{k}': v for k, v in param_dist_rf.items()}
#             else:
#                 base_model = Pipeline([
#                     ('imputer', SimpleImputer(strategy = 'median')),
#                     ('regressor',     XGBRegressor(random_state=random_state_num,
#                                     n_jobs=n_cores,
#                                     verbosity = 0)),    
#                 ])
#                 param_dist = {f'regressor__{k}': v for k, v in param_dist_xgb.items()}

#             search_cv = RandomizedSearchCV(
#                 estimator = base_model,
#                 param_distributions = param_dist,
#                 n_iter = 30,
#                 cv = 3,
#                 verbose = 1,
#                 n_jobs = n_cores,
#                 scoring = 'neg_root_mean_squared_error'
#             )
#             search_cv.fit(X_train, y_train)
#             best_model = search_cv.best_estimator_
#             rscv_results_df = pd.DataFrame(search_cv.cv_results_)
#             rscv_path  = os.path.join(composite_model_path, f"RSCV_compositeRegressor_{9.0}_{season}_{model_type}.csv")
#             rscv_results_df.to_csv(rscv_path)

#             print(f"\n  [Cluster {season} | {model_type.upper()}]")
#             print(f"    Best params : {search_cv.best_params_}")
#             print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
#             print(f"RSCV results saved to : {rscv_path}")
                
#             model_path = os.path.join(composite_model_path, f"compositeRegressor_{9.0}_{season}_{model_type}.pkl")
#             joblib.dump(best_model, model_path)
#             print(f"Model  {model_type} saved to {model_path}")

#     elif season in ['pre_monsoon', 'winter']:
#         continue  # Skip other seasons for now 
#         param_dist_rf, param_dist_xgb = get_param_grids()
#         for model_type in ['rf', 'xgb']:
#             if model_type == 'rf':
#                 base_model = Pipeline([
#                     ('imputer', SimpleImputer(strategy = 'median')),
#                     ('regressor', RandomForestRegressor(random_state = random_state_num,
#                         n_jobs = n_cores))
#                 ])
#                 param_dist = {f'regressor__{k}': v for k, v in param_dist_rf.items()}
#             else:
#                 base_model = Pipeline([
#                     ('imputer', SimpleImputer(strategy = 'median')),
#                     ('regressor',     XGBRegressor(random_state=random_state_num,
#                                     n_jobs=n_cores,
#                                     verbosity = 0)),    
#                 ])
#                 param_dist = {f'regressor__{k}': v for k, v in param_dist_xgb.items()}

#             search_cv = RandomizedSearchCV(
#                 estimator = base_model,
#                 param_distributions = param_dist,
#                 n_iter = 30,
#                 cv = 3,
#                 verbose = 1,
#                 n_jobs = 5,
#                 scoring = 'neg_root_mean_squared_error'
#             )
#             search_cv.fit(X_train, y_train)
#             best_model = search_cv.best_estimator_
#             rscv_results_df = pd.DataFrame(search_cv.cv_results_)
#             rscv_path  = os.path.join(composite_model_path, f"RSCV_compositeRegressor_{9.0}_{season}_{model_type}.csv")
#             rscv_results_df.to_csv(rscv_path)

#             print(f"\n  [Cluster {season} | {model_type.upper()}]")
#             print(f"    Best params : {search_cv.best_params_}")
#             print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
#             print(f"RSCV results saved to : {rscv_path}")
                
#             model_path = os.path.join(composite_model_path, f"compositeRegressor_{9.0}_{season}_{model_type}.pkl")
#             joblib.dump(best_model, model_path)
#             print(f"Model  {model_type} saved to {model_path}")

#     else:
# #        continue  # Skip other seasons for now 
#         param_dist_rf, param_dist_xgb = get_param_grids()
#         for model_type in ['rf', 'xgb']:
#             if model_type == 'rf':
#                 base_model = Pipeline([
#                     ('imputer', SimpleImputer(strategy = 'median')),
#                     ('regressor', RandomForestRegressor(random_state = random_state_num,
#                         n_jobs = n_cores))
#                 ])
#                 param_dist = {f'regressor__{k}': v for k, v in param_dist_rf.items()}
#             else:
#                 base_model = Pipeline([
#                     ('imputer', SimpleImputer(strategy = 'median')),
#                     ('regressor',     XGBRegressor(random_state=random_state_num,
#                                     n_jobs=n_cores,
#                                     verbosity = 0)),    
#                 ])
#                 param_dist = {f'regressor__{k}': v for k, v in param_dist_xgb.items()}

#             search_cv = RandomizedSearchCV(
#                 estimator = base_model,
#                 param_distributions = param_dist,
#                 n_iter = 30,
#                 cv = 3,
#                 verbose = 1,
#                 n_jobs = n_cores,
#                 scoring = 'neg_root_mean_squared_error'
#             )
#             search_cv.fit(X_train, y_train)
#             best_model = search_cv.best_estimator_
#             rscv_results_df = pd.DataFrame(search_cv.cv_results_)
#             rscv_path  = os.path.join(composite_model_path, f"RSCV_compositeRegressor_{9.0}_{season}_{model_type}.csv")
#             rscv_results_df.to_csv(rscv_path)

#             print(f"\n  [Cluster {season} | {model_type.upper()}]")
#             print(f"    Best params : {search_cv.best_params_}")
#             print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
#             print(f"RSCV results saved to : {rscv_path}")
                
#             model_path = os.path.join(composite_model_path, f"compositeRegressor_{9.0}_{season}_{model_type}.pkl")
#             joblib.dump(best_model, model_path)
#             print(f"Model  {model_type} saved to {model_path}")
        

In [6]:
# # 2 Design the chained model
# # For classifier : XGB with seasonal stratification 
# # For regressor : RF with seasonal stratification

# test_df['insitu_prec_class_pred'] = np.nan
# test_df['insitu_prec_pred_a1'] = np.nan
# test_df['insitu_prec_pred_a2'] = np.nan

# for season in season_list:
#     print(f"\nSeason: {season}")

#     test_df_season = test_df[test_df['season'] == season].copy()

#     #Alternative 1: Chained model
#     classifier_model_file = os.path.join(classifier_model_path, f"9.0_{season}_xgb.pkl")
#     classifier_model = joblib.load(classifier_model_file)
#     classification_features = test_df_season[predictor_features].drop(columns=['season', 'season_num'], errors='ignore')
#     test_df_season['insitu_prec_class_pred'] = classifier_model.predict(classification_features)

#     is_wet = test_df_season['insitu_prec_class_pred'] == 1
    
#     test_df_season_wet = test_df_season[is_wet].copy()
#     test_df_season_dry = test_df_season[~is_wet].copy()

#     test_df_season_dry['insitu_prec_pred_a1'] = 0.0
#     regression_model_file = os.path.join(regression_model_path, f"9.0_{season}_rf.pkl")
#     regression_model = joblib.load(regression_model_file)

#     regression_features = test_df_season_wet[predictor_features].drop(columns=['season', 'season_num'], errors='ignore')
#     preds = regression_model.predict(regression_features)
#     test_df_season_wet['insitu_prec_pred_a1'] = preds
#     test_df_season.update(test_df_season_wet)
#     test_df_season.update(test_df_season_dry)

#     test_df.update(test_df_season)

#     #Alternative 2: Single model
#     composite_model_file = os.path.join(composite_model_path, f"compositeRegressor_{9.0}_{season}_rf.pkl")
#     composite_model = joblib.load(composite_model_file)
#     composite_features = test_df[predictor_features].drop(columns=['season', 'season_num'], errors='ignore')
#     test_df['insitu_prec_pred_a2'] = composite_model.predict(composite_features)

# test_df['insitu_prec_pred_a1'] = test_df['insitu_prec_pred_a1'].clip(lower=0)
# test_df['insitu_prec_pred_a2'] = test_df['insitu_prec_pred_a2'].clip(lower=0)

In [7]:
# metric_single_model = compute_metrics_regression(test_df['insitu_prec'], test_df['insitu_prec_pred_a2'], label="Single model")
# metric_chained_model = compute_metrics_regression(test_df['insitu_prec'], test_df['insitu_prec_pred_a1'], label="Chained model")

In [8]:
# # %%
# # Bias correction model comparison — simplified
# # 4 indicators, one table, one plot.
# # Single = insitu_prec_pred_a2 | Chained = insitu_prec_pred_a1

# import warnings
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from sklearn.metrics import mean_squared_error, r2_score
# from scipy.stats import spearmanr

# WET   = 0.1    # mm wet-day threshold
# HEAVY = 10.0   # mm threshold for heavy+ category

# MODELS = {"Single": "insitu_prec_pred_a2", "Chained": "insitu_prec_pred_a1"}

# INTENSITY = [
#     ("Dry",        0.0,   WET),
#     ("Light",      WET,   2.5),
#     ("Moderate",   2.5,  HEAVY),
#     ("Heavy",     HEAVY,  35.0),
#     ("Very heavy", 35.0,  np.inf),
# ]

# # ── 4 indicators ─────────────────────────────────────────────────────────────

# def ets(obs, pred):
#     """Equitable Threat Score — wet-day detection (0 = no skill, 1 = perfect)."""
#     o = obs >= WET;  p = pred >= WET
#     H, M, F = np.sum(o & p), np.sum(o & ~p), np.sum(~o & p)
#     N = len(obs)
#     Hr = (H + M) * (H + F) / N
#     return (H - Hr) / (H + M + F - Hr) if (H + M + F - Hr) > 0 else np.nan

# def rmse_heavy(obs, pred):
#     """RMSE on heavy+ events only (obs ≥ 10 mm) — where volume matters most."""
#     mask = obs >= HEAVY
#     return np.sqrt(mean_squared_error(obs[mask], pred[mask])) if mask.sum() > 0 else np.nan

# def mean_bias(obs, pred):
#     """Mean bias in mm (pred − obs) — always defined, sign shows direction."""
#     return float(np.mean(pred - obs))

# def rmse_by_intensity(obs, pred):
#     """RMSE for each intensity band."""
#     out = {}
#     for name, lo, hi in INTENSITY:
#         mask = (obs >= lo) & (obs < hi)
#         out[name] = np.sqrt(mean_squared_error(obs[mask], pred[mask])) if mask.sum() >= 5 else np.nan
#     return out

# # ── compute for both models ───────────────────────────────────────────────────

# obs = test_df["insitu_prec"].values

# results = {}
# for model_name, col in MODELS.items():
#     pred = test_df[col].values
#     results[model_name] = {
#         "ETS (wet-day)":     round(ets(obs, pred), 4),
#         "RMSE heavy+ (mm)":  round(rmse_heavy(obs, pred), 3),
#         "Mean bias (mm)":    round(mean_bias(obs, pred), 3),
#         "R²":                round(r2_score(obs, pred), 4),
#     }
#     results[model_name]["_intensity"] = rmse_by_intensity(obs, pred)

# # ── summary table ─────────────────────────────────────────────────────────────

# indicators = ["R²", "ETS (wet-day)", "RMSE heavy+ (mm)", "Mean bias (mm)"]
# higher_better = {"R²": True, "ETS (wet-day)": True,
#                  "RMSE heavy+ (mm)": False, "Mean bias (mm)": None}  # None = closer to 0

# rows = []
# for ind in indicators:
#     s_val = results["Single"][ind]
#     c_val = results["Chained"][ind]
#     hb    = higher_better[ind]
#     if hb is True:
#         winner = "Single" if s_val > c_val else "Chained" if c_val > s_val else "Tie"
#     elif hb is False:
#         winner = "Single" if s_val < c_val else "Chained" if c_val < s_val else "Tie"
#     else:  # closest to zero
#         winner = "Single" if abs(s_val) < abs(c_val) else "Chained" if abs(c_val) < abs(s_val) else "Tie"
#     rows.append({"Indicator": ind, "Single": s_val, "Chained": c_val, "Better": winner})

# summary = pd.DataFrame(rows)
# print("\n── Overall comparison ──────────────────────────────────────────────")
# print(summary.to_string(index=False))

# # ── RMSE by intensity ─────────────────────────────────────────────────────────

# int_rows = []
# for name, _, _ in INTENSITY:
#     s = results["Single"]["_intensity"][name]
#     c = results["Chained"]["_intensity"][name]
#     winner = "Single" if (s or np.inf) < (c or np.inf) else \
#              "Chained" if (c or np.inf) < (s or np.inf) else "Tie"
#     int_rows.append({"Category": name,
#                      "Single RMSE":  round(s, 3) if s else np.nan,
#                      "Chained RMSE": round(c, 3) if c else np.nan,
#                      "Better": winner})

# intensity_summary = pd.DataFrame(int_rows)
# print("\n── RMSE by intensity category ──────────────────────────────────────")
# print(intensity_summary.to_string(index=False))

# # ── plot ──────────────────────────────────────────────────────────────────────

# fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# fig.suptitle("Single vs Chained bias correction model", fontsize=13, fontweight="bold")

# colors = ["steelblue", "darkorange"]

# # Left: 4 headline indicators (normalised to [0,1] for display on one axis)
# ax = axes[0]
# x  = np.arange(len(indicators))
# w  = 0.35
# for i, (model_name, _) in enumerate(MODELS.items()):
#     vals = [results[model_name][ind] for ind in indicators]
#     ax.bar(x + i * w, vals, w, label=model_name, color=colors[i], alpha=0.85)
# ax.set_xticks(x + w / 2)
# ax.set_xticklabels(indicators, rotation=15, ha="right", fontsize=9)
# ax.axhline(0, color="black", linewidth=0.7)
# ax.set_title("4 headline indicators\n(R², ETS: higher better | RMSE, bias: lower/nearer 0)")
# ax.legend(); ax.grid(True, alpha=0.3, axis="y")

# # Right: RMSE by intensity category
# ax = axes[1]
# cats = [r["Category"] for r in int_rows]
# xc   = np.arange(len(cats))
# for i, model_name in enumerate(MODELS):
#     vals = [results[model_name]["_intensity"][c] for c in cats]
#     ax.bar(xc + i * w, vals, w, label=model_name, color=colors[i], alpha=0.85)
# ax.set_xticks(xc + w / 2)
# ax.set_xticklabels(cats, rotation=15, ha="right", fontsize=9)
# ax.set_ylabel("RMSE (mm)"); ax.set_title("RMSE by intensity category")
# ax.legend(); ax.grid(True, alpha=0.3, axis="y")

# # ── per-season and per-cluster comparison ──────────────────────────────────

# def compute_indicator_dict(obs, pred):
#     """Compute the same 4 headline indicators for a given obs/pred slice."""
#     return {
#         "R²":                round(r2_score(obs, pred), 4) if len(obs) > 1 else np.nan,
#         "ETS (wet-day)":     round(ets(obs, pred), 4),
#         "RMSE heavy+ (mm)":  round(rmse_heavy(obs, pred), 3),
#         "Mean bias (mm)":    round(mean_bias(obs, pred), 3),
#         "n":                 len(obs),
#     }

# def build_group_comparison(df, group_col, group_values):
#     """Compute Single vs Chained indicators for each value of group_col."""
#     rows = []
#     for g in group_values:
#         sub = df[df[group_col] == g]
#         obs_g = sub["insitu_prec"].values
#         for model_name, col in MODELS.items():
#             pred_g = sub[col].values
#             d = compute_indicator_dict(obs_g, pred_g)
#             d[group_col] = g
#             d["Model"] = model_name
#             rows.append(d)
#     return pd.DataFrame(rows)

# # --- by season ---
# season_comparison = build_group_comparison(test_df, "season", season_list)
# season_comparison = season_comparison[["season", "Model", "n", "R²", "ETS (wet-day)",
#                                         "RMSE heavy+ (mm)", "Mean bias (mm)"]]
# print("\n── Comparison by season ────────────────────────────────────────────")
# print(season_comparison.to_string(index=False))

# # --- by cluster ---
# cluster_comparison = build_group_comparison(test_df, "cluster", cluster_list)
# cluster_comparison = cluster_comparison[["cluster", "Model", "n", "R²", "ETS (wet-day)",
#                                           "RMSE heavy+ (mm)", "Mean bias (mm)"]]
# print("\n── Comparison by cluster ───────────────────────────────────────────")
# print(cluster_comparison.to_string(index=False))

# # ── plots: per-season and per-cluster ───────────────────────────────────────

# def plot_group_comparison(group_df, group_col, group_values, title):
#     fig, axes = plt.subplots(2, 2, figsize=(12, 9))
#     fig.suptitle(title, fontsize=13, fontweight="bold")
#     metrics_to_plot = ["R²", "ETS (wet-day)", "RMSE heavy+ (mm)", "Mean bias (mm)"]

#     x = np.arange(len(group_values))
#     w = 0.35

#     for ax, metric in zip(axes.flat, metrics_to_plot):
#         for i, model_name in enumerate(MODELS):
#             vals = [group_df[(group_df[group_col] == g) & (group_df["Model"] == model_name)][metric].values[0]
#                     for g in group_values]
#             ax.bar(x + i * w, vals, w, label=model_name, color=colors[i], alpha=0.85)
#         ax.set_xticks(x + w / 2)
#         ax.set_xticklabels(group_values, rotation=30, ha="right", fontsize=8)
#         ax.axhline(0, color="black", linewidth=0.6)
#         ax.set_title(metric, fontsize=10)
#         ax.grid(True, alpha=0.3, axis="y")

#     axes.flat[0].legend()
#     plt.tight_layout(rect=[0, 0, 1, 0.96])
#     return fig

# fig_season = plot_group_comparison(season_comparison, "season", season_list,
#                                     "Single vs Chained — by season")
# fig_cluster = plot_group_comparison(cluster_comparison, "cluster", cluster_list,
#                                      "Single vs Chained — by cluster")

# plt.show()

In [ ]:
# Retrain season only XGB-classifier and RF-regressor with recursive feature elimination and fixed parameters

def get_fixed_params(season, classifier=True):
    if classifier:
        model_name = 'xgb'
        rscv_file = fr"..\Data\Processed\Bias_correction_daily\Classifier_model\rscv_results\{9.0}_{season}_{model_name}.csv"
        rscv_results_df = pd.read_csv(rscv_file)
        best_params_str = rscv_results_df[rscv_results_df["rank_test_score"] == 1]["params"].values[0]
    else:
        model_name = 'rf'
        rscv_file = fr"..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\{9.0}_{season}_{model_name}.csv"
        rscv_results_df = pd.read_csv(rscv_file)
        best_params_str = rscv_results_df[rscv_results_df["rank_test_score"] == 1]["params"].values[0]

    best_params = ast.literal_eval(best_params_str)
    
    clean_params = {key.split('__')[-1]: value for key, value in best_params.items()}
    
    return clean_params

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"



MAX_RFE_ESTIMATORS   = 300   
MIN_FEATURES_TO_SELECT = 5
RFE_STEP              = 2   

for season in season_list:
    print(f"\nRetraining models for season: {season}")

    # Load training data for the season
    season_train_df = train_df[train_df['season'] == season].copy()
    season_train_df['insitu_prec_class'] = (season_train_df['insitu_prec'] >= 0.1).astype(int)
    X_train = season_train_df[predictor_features].drop(columns=['season', 'season_num'], errors='ignore').copy()
    X_train = X_train.astype('float32')  
    y_train_class = season_train_df[target_var_classification].copy()
    y_train_reg = season_train_df[target_var_regression].copy()

    feature_order = X_train.columns.tolist()

    # classifier
    best_params_class = get_fixed_params(season, classifier=True)

    proxy_classifier = XGBClassifier(
        random_state=random_state_num,
        n_jobs=n_cores,
        verbosity=0,
        tree_method='hist',
        n_estimators=min(best_params_class.get('n_estimators', 300), MAX_RFE_ESTIMATORS),
        max_depth=best_params_class.get('max_depth', 6),
    )

    rfecv_classifier = RFECV(
        estimator=proxy_classifier,
        step=RFE_STEP,
        min_features_to_select=MIN_FEATURES_TO_SELECT,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state_num),
        scoring='f1',
        n_jobs=n_cores
    )

    class_selector = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('rfecv_class', rfecv_classifier)
    ])
    class_selector.fit(X_train, y_train_class)

    class_support_mask = class_selector.named_steps['rfecv_class'].support_
    classifier_selected_features = [f for f, keep in zip(feature_order, class_support_mask) if keep]
    print(f"  [Classifier] Optimal features selected: {len(classifier_selected_features)} / {X_train.shape[1]}")

    del rfecv_classifier, proxy_classifier, class_selector
    gc.collect()

    final_classifier = XGBClassifier(
        random_state=random_state_num,
        n_jobs=n_cores,
        verbosity=0,
        tree_method='hist',
        **best_params_class
    )
    classifier_model = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', final_classifier)
    ])
    classifier_model.fit(X_train[classifier_selected_features], y_train_class)

    # 2. REGRESSOR 
    best_params_reg = get_fixed_params(season, classifier=False)

    proxy_regressor = RandomForestRegressor(
        random_state=random_state_num,
        n_jobs=n_cores,
        n_estimators=min(best_params_reg.get('n_estimators', 300), MAX_RFE_ESTIMATORS),
        max_depth=best_params_reg.get('max_depth') or 15,  # cap depth even if tuned value is None
    )

    rfecv_regressor = RFECV(
        estimator=proxy_regressor,
        step=RFE_STEP,
        min_features_to_select=MIN_FEATURES_TO_SELECT,
        cv=KFold(n_splits=5, shuffle=True, random_state=random_state_num),
        scoring='neg_mean_squared_error',
        n_jobs=n_cores
    )

    reg_selector = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('rfecv_reg', rfecv_regressor)
    ])
    reg_selector.fit(X_train, y_train_reg)

    reg_support_mask = reg_selector.named_steps['rfecv_reg'].support_
    regressor_selected_features = [f for f, keep in zip(feature_order, reg_support_mask) if keep]
    print(f"  [Regressor] Optimal features selected: {len(regressor_selected_features)} / {X_train.shape[1]}")

    del rfecv_regressor, proxy_regressor, reg_selector
    gc.collect()

    final_regressor = RandomForestRegressor(
        random_state=random_state_num,
        n_jobs=n_cores,
        **best_params_reg
    )
    regressor_model = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', final_regressor)
    ])
    regressor_model.fit(X_train[regressor_selected_features], y_train_reg)


    joblib.dump(classifier_model, os.path.join(final_model_path, f"Classifier_{season}_xgb.pkl"))
    joblib.dump(regressor_model, os.path.join(final_model_path, f"Regressor_{season}_rf.pkl"))

    metadata = {
        'season': season,
        'full_input_feature_order': feature_order,
        'classifier_selected_features': classifier_selected_features,
        'regressor_selected_features': regressor_selected_features,
    }
    with open(os.path.join(final_model_path, f"feature_metadata_{season}.json"), 'w') as f:
        json.dump(metadata, f, indent=2)

    # Free memory before the next season
    del (season_train_df, X_train, y_train_class, y_train_reg,
         classifier_model, regressor_model, final_classifier, final_regressor)
    gc.collect()

    print(f"  Saved models + feature metadata for season: {season}")


Retraining models for season: winter
  [Classifier] Optimal features selected: 16 / 28


KeyboardInterrupt: 

In [5]:
 
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, r2_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import joblib
import json
import os
import gc
import warnings
 
# --- Configuration ---
N_REPEATS   = 10    # Repeats per permutation
RANDOM_SEED = random_state_num  # Ensure this is defined in your main script
MAX_SAMPLES = 50000  # Cap rows for permutation to prevent OOM
PERM_JOBS   = 1
# MEM: dropped default from 2 -> 1. sklearn's permutation_importance pickles/
# copies the fitted model into every worker process when n_jobs > 1, so with
# a large RF/XGB model this roughly multiplies peak RAM by PERM_JOBS on top
# of the data duplication. Only raise this if you have headroom to spare.
 
importance_out_dir = os.path.join(final_model_path, "importance_metrics")
diag_out_dir = os.path.join(importance_out_dir, "plots")
os.makedirs(importance_out_dir, exist_ok=True)
os.makedirs(diag_out_dir, exist_ok=True)
 
# Define categories strictly based on your `predictor_features` list
FEATURE_CATEGORIES = {
    'Precipitation_datasets': ['era5l', 'chirps', 'har', 'persiann_prec', 'mswep_prec'],
    'GLDAS_Weather':          ['gldas_air_temp', 'gldas_lwdown', 'gldas_specific_humidity',
                               'gldas_surface_pressure', 'gldas_swdown', 'gldas_wind_speed'],
    'Topographic':            ['elev_mean', 'elev_std', 'elev_max', 'elev_min',
                               'slope_mean', 'aspect_mean', 'elev_smoothed', 'local_relief'],
    'Fetch':                  ['fetch_1000m_dist', 'fetch_1000m_dir',
                               'fetch_500m_dist', 'fetch_500m_dir'],
    'Spatial':                ['y', 'x'],
    'Temporal':               ['doy_sin', 'doy_cos'],
    'Cluster':                ['cluster'],
}
 
feature_to_category = {f: cat for cat, feats in FEATURE_CATEGORIES.items() for f in feats}
 
CATEGORY_COLORS = {
    'Precipitation_datasets': '#2166AC',
    'GLDAS_Weather':          '#74ADD1',
    'Topographic':            '#4DAC26',
    'Fetch':                  '#A6DBA0',
    'Spatial':                '#762A83',
    'Temporal':               '#C2A5CF',
    'Cluster':                '#E08214',
}
 
models_to_evaluate = [
    {'type': 'Classifier', 'suffix': 'xgb', 'scoring': 'f1', 'metric_func': f1_score, 'target': 'insitu_prec_class'},
    {'type': 'Regressor',  'suffix': 'rf',  'scoring': 'r2', 'metric_func': r2_score, 'target': 'insitu_prec'}
]
 
 
def get_season_sample_index(df, season, max_samples, seed):
    """
    MEM: Pick WHICH rows we want before touching any feature columns.
    df['season'] == season is a cheap boolean scan (no new column data is
    materialized). We then draw at most `max_samples` row labels from that
    index. Only those specific rows are ever pulled out of df, regardless of
    how many rows the season actually has.
    """
    season_index = df.index[df['season'] == season]
    n_available = len(season_index)
    if n_available > max_samples:
        rng = np.random.RandomState(seed)
        chosen = rng.choice(n_available, size=max_samples, replace=False)
        return season_index[chosen], n_available
    return season_index, n_available
 
 
# --- Loop through seasons and models ---
for season in season_list:
    print(f"\n==========================================================")
    print(f" Calculating Feature Importance for Season: {season}")
    print(f"==========================================================")
 
    metadata_path = os.path.join(final_model_path, f"feature_metadata_{season}.json")
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
 
    all_needed_feats = set(metadata['classifier_selected_features'] + metadata['regressor_selected_features'])
    cols_to_keep = list(all_needed_feats) + ['insitu_prec', 'season']
    available_cols = [c for c in cols_to_keep if c in test_df.columns]
 
    # 1. Decide the sample first, load second.
    sample_index, n_available = get_season_sample_index(test_df, season, MAX_SAMPLES, RANDOM_SEED)
    if n_available > MAX_SAMPLES:
        print(f"  Season has {n_available} rows; sampling {MAX_SAMPLES} before loading feature columns.")
 
    # MEM: this is now the ONLY point where season data is materialized, and
    # it's capped at MAX_SAMPLES rows no matter how big the season is.
    # .loc[] with an explicit row-label list already returns a new object,
    # so the extra .copy() from the original version is unnecessary.
    season_test_df = test_df.loc[sample_index, available_cols]
    season_test_df['insitu_prec_class'] = (season_test_df['insitu_prec'] >= 0.1).astype(np.int8)
 
    for mod_info in models_to_evaluate:
        m_type = mod_info['type']
        print(f"\n── Evaluating {m_type} ({season}) ──")
 
        selected_features = metadata[f'{m_type.lower()}_selected_features']
        target_var = mod_info['target']
 
        # MEM: build a raw float32 numpy array once, and reuse it for
        # baseline prediction, individual permutation importance, AND the
        # category block importance below — instead of holding both a
        # DataFrame (X_test_sel) and a separate numpy copy (X_test_np) as
        # the original script did. Passing a numpy array (not a DataFrame)
        # into permutation_importance also lets joblib's loky backend
        # memory-map it across worker processes instead of pickling copies
        # when n_jobs > 1.
        X_test = season_test_df[selected_features].to_numpy(dtype='float32')
        y_test = season_test_df[target_var].to_numpy()
 
        warnings.filterwarnings("ignore", message="X does not have valid feature names")
 
        model_file = f"{m_type}_{season}_{mod_info['suffix']}.pkl"
        model = joblib.load(os.path.join(final_model_path, model_file))
 
        baseline_pred = model.predict(X_test)
        baseline_score = mod_info['metric_func'](y_test, baseline_pred)
        score_name = mod_info['scoring'].upper()
 
        # ---------------------------------------------------------
        # A. INDIVIDUAL PERMUTATION IMPORTANCE
        # ---------------------------------------------------------
        print("  Calculating Individual Feature Importance...")
        perm = permutation_importance(
            model, X_test, y_test,
            n_repeats=N_REPEATS,
            random_state=RANDOM_SEED,
            n_jobs=PERM_JOBS,
            scoring=mod_info['scoring']
        )
 
        indiv_rows = []
        for i, feat in enumerate(selected_features):
            indiv_rows.append({
                'season': season,
                'model_type': m_type,
                'feature': feat,
                'category': feature_to_category.get(feat, 'Other'),
                'importance_mean': round(perm.importances_mean[i], 6),
                'importance_std':  round(perm.importances_std[i], 6),
            })
 
        individual_df = pd.DataFrame(indiv_rows).sort_values('importance_mean', ascending=False).reset_index(drop=True)
        individual_df['rank'] = individual_df['importance_mean'].rank(ascending=False, method='min').astype(int)
        individual_df.to_csv(os.path.join(importance_out_dir, f"perm_imp_individual_{season}_{m_type}.csv"), index=False)
 
        # ---------------------------------------------------------
        # B. CATEGORY (BLOCK) PERMUTATION IMPORTANCE
        # ---------------------------------------------------------
        print("  Calculating Category Block Importance...")
        category_rows = []
 
        for cat_name, cat_features in FEATURE_CATEGORIES.items():
            cat_features_present = [f for f in cat_features if f in selected_features]
            if not cat_features_present:
                continue
 
            cat_indices = [selected_features.index(f) for f in cat_features_present]
            repeat_scores = []
            rng = np.random.default_rng(RANDOM_SEED)
 
            # MEM: only copy the small block of columns being shuffled
            # (e.g. 2-6 columns), not the entire X_test matrix on every one
            # of the N_REPEATS x len(FEATURE_CATEGORIES) iterations like the
            # original `X_permuted = X_test_np.copy()` did. We permute
            # X_test in place and restore the original block afterward.
            original_block = X_test[:, cat_indices].copy()
            try:
                for _ in range(N_REPEATS):
                    perm_idx = rng.permutation(len(X_test))
                    X_test[:, cat_indices] = original_block[perm_idx]
                    permuted_pred = model.predict(X_test)
                    repeat_scores.append(mod_info['metric_func'](y_test, permuted_pred))
            finally:
                # Always restore, even if predict() raises, so later
                # categories (and the collinearity check) see clean data.
                X_test[:, cat_indices] = original_block
                del original_block
 
            drop_mean = baseline_score - np.mean(repeat_scores)
            drop_std = np.std(repeat_scores)
 
            category_rows.append({
                'season': season,
                'model_type': m_type,
                'category': cat_name,
                'n_features_used': len(cat_features_present),
                'features': ', '.join(cat_features_present),
                f'baseline_{score_name}': round(baseline_score, 6),
                'importance_mean': round(drop_mean, 6),
                'importance_std': round(drop_std, 6),
            })
 
        category_df = pd.DataFrame(category_rows).sort_values('importance_mean', ascending=False).reset_index(drop=True)
        category_df['rank'] = category_df['importance_mean'].rank(ascending=False, method='min').astype(int)
        category_df.to_csv(os.path.join(importance_out_dir, f"perm_imp_category_{season}_{m_type}.csv"), index=False)
 
        # ---------------------------------------------------------
        # C. PLOTTING
        # ---------------------------------------------------------
        fig, ax = plt.subplots(figsize=(10, max(6, len(individual_df) * 0.35)))
        colors = [CATEGORY_COLORS.get(cat, '#AAAAAA') for cat in individual_df['category']]
 
        ax.barh(individual_df['feature'], individual_df['importance_mean'], xerr=individual_df['importance_std'],
                color=colors, edgecolor='white', linewidth=0.4,
                error_kw=dict(elinewidth=0.8, ecolor='#444444', capsize=2), height=0.7)
 
        ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
        ax.set_xlabel(f'Permutation importance (mean {score_name} drop)', fontsize=11)
        ax.set_title(f'Individual Feature Importance — {m_type} ({season})\n(test set, {N_REPEATS} repeats)', fontsize=12, pad=10)
        ax.invert_yaxis()
 
        legend_handles = [mpatches.Patch(color=c, label=cat.replace('_', ' '))
                          for cat, c in CATEGORY_COLORS.items() if cat in individual_df['category'].values]
        ax.legend(handles=legend_handles, fontsize=9, loc='lower right', framealpha=0.85, title='Category')
 
        plt.tight_layout()
        fig.savefig(os.path.join(diag_out_dir, f"plot_indiv_{season}_{m_type}.png"), dpi=150, bbox_inches='tight')
        fig.clf()
        plt.close(fig)
 
        fig, ax = plt.subplots(figsize=(8, max(4, len(category_df) * 0.6)))
        cat_colors = [CATEGORY_COLORS.get(cat, '#AAAAAA') for cat in category_df['category']]
 
        ax.barh(category_df['category'].str.replace('_', ' '), category_df['importance_mean'], xerr=category_df['importance_std'],
                color=cat_colors, edgecolor='white', linewidth=0.4,
                error_kw=dict(elinewidth=0.8, ecolor='#444444', capsize=2), height=0.6)
 
        for i, (_, row) in enumerate(category_df.iterrows()):
            ax.text(max(row['importance_mean'] + row['importance_std'] + 0.001, 0.001), i,
                    f"n={row['n_features_used']}", va='center', fontsize=9, color='#333333')
 
        ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
        ax.set_xlabel(f'Block permutation importance (mean {score_name} drop)', fontsize=11)
        ax.set_title(f'Feature Category Importance — {m_type} ({season})\n(baseline {score_name}={baseline_score:.4f})', fontsize=12, pad=10)
        ax.invert_yaxis()
 
        plt.tight_layout()
        fig.savefig(os.path.join(diag_out_dir, f"plot_cat_{season}_{m_type}.png"), dpi=150, bbox_inches='tight')
        fig.clf()
        plt.close(fig)
 
        # ---------------------------------------------------------
        # D. COLLINEARITY CHECK
        # ---------------------------------------------------------
        # MEM: wrap the numpy array with copy=False so this doesn't
        # duplicate the sampled feature matrix just to compute ranks.
        corr = pd.DataFrame(X_test, columns=selected_features, copy=False).corr(method='spearman').abs()
        pairs = (
            corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            .stack().reset_index()
            .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b', 0: 'spearman_r'})
            .query('spearman_r > 0.85')
            .sort_values('spearman_r', ascending=False)
            .reset_index(drop=True)
        )
        if not pairs.empty:
            pairs.to_csv(os.path.join(importance_out_dir, f"collinear_pairs_{season}_{m_type}.csv"), index=False)
            print(f"  Flagged {len(pairs)} highly correlated feature pairs (Spearman > 0.85).")
 
        del model, X_test, y_test, perm, corr, pairs
        gc.collect()
 
    del season_test_df
    gc.collect()
 
    print(f"  Finished importance metrics for {season}. Outputs saved to {importance_out_dir}\n")
 
warnings.resetwarnings()


 Calculating Feature Importance for Season: winter

── Evaluating Classifier (winter) ──
  Calculating Individual Feature Importance...
  Calculating Category Block Importance...
  Flagged 1 highly correlated feature pairs (Spearman > 0.85).

── Evaluating Regressor (winter) ──
  Calculating Individual Feature Importance...
  Calculating Category Block Importance...
  Flagged 2 highly correlated feature pairs (Spearman > 0.85).
  Finished importance metrics for winter. Outputs saved to ..\Data\Processed\Bias_correction_daily\Final_model\importance_metrics


 Calculating Feature Importance for Season: pre_monsoon

── Evaluating Classifier (pre_monsoon) ──
  Calculating Individual Feature Importance...
  Calculating Category Block Importance...
  Flagged 12 highly correlated feature pairs (Spearman > 0.85).

── Evaluating Regressor (pre_monsoon) ──
  Calculating Individual Feature Importance...
  Calculating Category Block Importance...
  Finished importance metrics for pre_monsoon. Out